# LifeMate — notebook đo thời gian

Chạy **Run All**. Kết quả ghi vào `/result/jupyter_submission.jsonl`: cùng `tool_calls` và `state_hash` với
`/result/submission.jsonl` của `predict.sh`, thêm `time` = số ms của từng mẫu, tính từ lúc tạo `ToolEnv`
(đọc audio, ASR, agent) tới `get_state_hash()`. Mọi tham số mặc định nằm trong `/code/predict.py`.

## Bước 1: seed

In [ ]:
import sys
sys.path.insert(0, "/code")

from lifemate.predict import seed_everything

SEED = 42
seed_everything(SEED)

## Bước 2: nạp model và tài nguyên

In [ ]:
from predict import CFG                 # /code/predict.py, same defaults as predict.sh
from lifemate.predict import Predictor

predictor = Predictor(CFG)              # LLM (vLLM) + ASR, both warmed up
model, asr_model = predictor.llm, predictor.asr
print(f"models loaded in {predictor.load_s:.1f}s")

## Bước 3: đọc test case trong /data

In [ ]:
from lifemate.data import read_jsonl
from lifemate.predict import find_split

sp = find_split(CFG.data, CFG.split)    # /data/<split>/
ToolEnv = predictor.open_split(sp)      # the split's own zaic_tools
test_cases = read_jsonl(sp.jsonl)
print(sp.split, len(test_cases), "test cases")

## Bước 4: dự đoán và ghi kết quả

In [ ]:
import json
import os
import time

all_result = []
for item in test_cases:
    t1 = time.perf_counter()
    tool_calls, state_hash = predictor.run_one(item)   # one ToolEnv -> ASR -> agent -> get_tool_calls / get_state_hash
    ms = int((time.perf_counter() - t1) * 1000)
    all_result.append({"id": item["id"], "tool_calls": tool_calls, "state_hash": state_hash, "time": ms})

os.makedirs("/result", exist_ok=True)
with open("/result/jupyter_submission.jsonl", "w", encoding="utf-8") as f:
    for r in all_result:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

times = sorted(r["time"] for r in all_result) or [0]
print(f"{len(all_result)} rows -> /result/jupyter_submission.jsonl, mean {sum(times) / len(times):.0f} ms, max {times[-1]} ms")